# Mouse Detection — Infrared Cage Footage (YOLO11n)
**Pipeline:** Roboflow dataset → grayscale conversion → offline IR-realistic augmentation → YOLO11n training → validation → real-video test → FPS/ms benchmark → ONNX export → fine-tuning template → tracking demo.

> Runtime: set **Runtime → Change runtime type → T4 GPU** before running.


## 1. Setup

In [ ]:
!pip install -q ultralytics albumentations roboflow

import ultralytics, torch
ultralytics.checks()
print("CUDA available:", torch.cuda.is_available())


Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)
CUDA available: True


## 2. Mount Google Drive
Used for two things: **(a)** training checkpoints are written to Drive so a Colab disconnect never loses progress, **(b)** the real cage frames (`cage_frames.zip`) are pulled from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/mouse_detect"     # checkpoints live here
ZIP_PATH    = "/content/drive/MyDrive/cage_frames.zip"  # <-- adjust if in a subfolder

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Checkpoints dir:", PROJECT_DIR)
print("cage_frames.zip found:", os.path.exists(ZIP_PATH))


Mounted at /content/drive
Checkpoints dir: /content/drive/MyDrive/mouse_detect
cage_frames.zip found: True


## 3. Download Roboflow dataset

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="z8u9lLnDVz2IznaV3MaW")
project = rf.workspace("mouse-dataset").project("mouse-v5ogi")
version = project.version(2)
dataset = version.download("yolov8")


DATA_DIR = dataset.location
print("Dataset at:", DATA_DIR)

!cat {DATA_DIR}/data.yaml


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to mouse-2 in yolov8:: 100%|██████████| 11900/11900 [00:06<00:00, 1775.98it/s]


Dataset at: /content/mouse-2
names:
- mouse
nc: 1
roboflow:
  license: CC BY 4.0
  project: mouse-v5ogi
  url: https://universe.roboflow.com/mouse-dataset/mouse-v5ogi/dataset/2
  version: 2
  workspace: mouse-dataset
test: ../test/images
train: ../train/images
val: ../valid/images


## 4. Force grayscale (all splits)
The deployment camera is IR/grayscale, so the model must never learn color cues. We convert **every** image (train/valid/test) to 3-channel grayscale in place.

In [ ]:
import cv2, glob, os
from pathlib import Path

def to_grayscale_inplace(img_dir):
    paths = glob.glob(os.path.join(img_dir, "*.*"))
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        cv2.imwrite(p, cv2.cvtColor(g, cv2.COLOR_GRAY2BGR))
    return len(paths)

for split in ["train", "valid", "test"]:
    d = os.path.join(DATA_DIR, split, "images")
    if os.path.isdir(d):
        n = to_grayscale_inplace(d)
        print(f"{split}: {n} images converted to grayscale")


train: 5314 images converted to grayscale
valid: 368 images converted to grayscale
test: 262 images converted to grayscale


## 5. Offline IR-realistic augmentation (train split only)
Photometric augmentations that mimic IR sensor behaviour: brightness/contrast drift, gamma, CLAHE, sensor noise, motion/Gaussian blur, compression artifacts. Geometric augmentation (flip, scale, translate, mosaic) is left to Ultralytics at train time.

`N_AUG_COPIES = 2` → final train set is 3× original size. Validation/test sets are **never** augmented.

In [ ]:
import albumentations as A
import numpy as np
import shutil

N_AUG_COPIES = 2  # augmented copies per original train image

ir_aug = A.Compose(
    [
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
        A.RandomGamma(gamma_limit=(70, 140), p=0.5),
        A.CLAHE(clip_limit=3.0, p=0.3),
        A.GaussNoise(p=0.4),
        A.OneOf(
            [A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=(3, 5))],
            p=0.3,
        ),
        A.ImageCompression(quality_range=(60, 95), p=0.3),
    ],
    bbox_params=A.BboxParams(
        format="yolo", label_fields=["class_labels"], min_visibility=0.2
    ),
)

train_img_dir = Path(DATA_DIR) / "train" / "images"
train_lbl_dir = Path(DATA_DIR) / "train" / "labels"

def read_yolo_labels(lbl_path):
    bboxes, classes = [], []
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                classes.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:]])
    return bboxes, classes

def write_yolo_labels(lbl_path, bboxes, classes):
    lines = [
        f"{c} " + " ".join(f"{v:.6f}" for v in b)
        for c, b in zip(classes, bboxes)
    ]
    lbl_path.write_text("\n".join(lines))

originals = sorted(p for p in train_img_dir.glob("*.*") if "_iraug" not in p.stem)
already_augmented = any("_iraug" in p.stem for p in train_img_dir.glob("*.*"))
print(f"Original train images: {len(originals)}")

if already_augmented:
    print("Augmented copies already present — skipping (safe re-run).")
    originals = []

created = 0
for img_path in originals:
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    lbl_path = train_lbl_dir / (img_path.stem + ".txt")
    bboxes, classes = read_yolo_labels(lbl_path)

    for i in range(N_AUG_COPIES):
        try:
            out = ir_aug(image=img, bboxes=bboxes, class_labels=classes)
        except Exception as e:
            print(f"skip {img_path.name}: {e}")
            continue
        new_stem = f"{img_path.stem}_iraug{i}"
        cv2.imwrite(str(train_img_dir / (new_stem + img_path.suffix)), out["image"])
        write_yolo_labels(
            train_lbl_dir / (new_stem + ".txt"),
            out["bboxes"],
            out["class_labels"],
        )
        created += 1

print(f"Created {created} augmented images. Train set now: "
      f"{len(list(train_img_dir.glob('*.*')))} images")


Got processor for bboxes, but no transform to process it.


Original train images: 5314
Created 10628 augmented images. Train set now: 15942 images


## 6. Train YOLO11n — interruption-safe
Three protection layers for Colab disconnects:
1. **Early stopping**: `patience=25` — training stops automatically if val mAP doesn't improve for 25 epochs.
2. **Checkpoints on Google Drive**: `last.pt` is updated every epoch + numbered checkpoints every 5 epochs (`save_period=5`), all written to `PROJECT_DIR` on Drive — they survive any disconnect.
3. **Auto-resume**: if the cell finds an unfinished run (`last.pt` exists), it resumes from the exact epoch where it stopped instead of starting over.

> After a disconnect: re-run cells 1–5 (env + dataset are ephemeral), then re-run this cell — it picks up where it left off.

In [ ]:
from ultralytics import YOLO
import os

MODEL    = "yolo11n.pt"   # swap to "yolov8n.pt" or "yolo11s.pt" for comparison runs
IMGSZ    = 640
EPOCHS   = 120
RUN_NAME = "yolo11n_pretrain"

LAST = f"{PROJECT_DIR}/{RUN_NAME}/weights/last.pt"
BEST = f"{PROJECT_DIR}/{RUN_NAME}/weights/best.pt"

# Did a previous run already finish? (resume on a completed run errors out)
def run_completed(run_dir):
    args = os.path.join(run_dir, "args.yaml")
    csv  = os.path.join(run_dir, "results.csv")
    if not (os.path.exists(args) and os.path.exists(csv)):
        return False
    import yaml
    n_epochs = yaml.safe_load(open(args)).get("epochs", EPOCHS)
    n_done = sum(1 for _ in open(csv)) - 1
    return n_done >= n_epochs

if os.path.exists(LAST) and not run_completed(f"{PROJECT_DIR}/{RUN_NAME}"):
    print(f"Unfinished run found — resuming from {LAST}")
    model = YOLO(LAST)
    results = model.train(resume=True)
elif os.path.exists(BEST) and run_completed(f"{PROJECT_DIR}/{RUN_NAME}"):
    print(f"Run already completed — using existing weights: {BEST}")
    model = YOLO(BEST)
else:
    print("Starting fresh training run")
    model = YOLO(MODEL)
    results = model.train(
        data=f"{DATA_DIR}/data.yaml",
        epochs=40,
        patience=10,          # early stopping on val mAP plateau
        save_period=5,        # extra numbered checkpoints every 5 epochs
        imgsz=IMGSZ,
        batch=-1,             # auto-fit batch to GPU memory
        optimizer="auto",
        lr0=0.01,
        cos_lr=True,
        # --- augmentation (geometric only; photometric done offline) ---
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.3,
        degrees=0.0,
        translate=0.1,
        scale=0.3,
        shear=0.0,
        perspective=0.0005,
        fliplr=0.5,
        flipud=0.0,
        mosaic=1.0,
        close_mosaic=5,
        mixup=0.0,
        copy_paste=0.0,
        # ---------------------------------------------------------------
        workers=2,
        seed=42,
        project=PROJECT_DIR,   # <-- checkpoints persist on Google Drive
        name=RUN_NAME,
        exist_ok=True,
    )

print("Best weights:", BEST)


Unfinished run found — resuming from /content/drive/MyDrive/mouse_detect/yolo11n_pretrain/weights/last.pt
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=5, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/mouse-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/mouse_detect/yolo11n_pretrain/weights/last.p

## 7. Validate on the held-out test split

In [ ]:
model = YOLO(BEST)

metrics = model.val(data=f"{DATA_DIR}/data.yaml", split="test", imgsz=IMGSZ)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

# Inspect PR curve / confusion matrix images:
import glob as g
print(g.glob(f"{PROJECT_DIR}/{RUN_NAME}/*.png")[:10])


Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 99.8±69.8 MB/s, size: 438.3 KB)
val: Scanning /content/mouse-2/test/labels... 262 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 262/262 558.1it/s 0.5s
val: New cache created: /content/mouse-2/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 7, len(boxes) = 291. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 3.7it/s 4.6s
                   all        262        291      0.967      0.924      0.971      0.778
Speed: 1.8ms preprocess, 8.0ms inference, 0.0ms loss, 1.7ms postprocess per image


## 8. (Optional) Model comparison — v8n vs 11n vs 11s
Runs shorter trainings for a fair quick comparison. Skip this cell if you only need the main model.

In [ ]:
COMPARE = False   # set True to run

if COMPARE:
    import pandas as pd, time

    rows = []
    for m in ["yolov8n.pt", "yolo11n.pt", "yolo11s.pt"]:
        name = m.replace(".pt", "") + "_cmp"
        mdl = YOLO(m)
        mdl.train(
            data=f"{DATA_DIR}/data.yaml", epochs=60, patience=15,
            imgsz=IMGSZ, batch=-1, hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
            degrees=0.0, fliplr=0.5, flipud=0.0, mosaic=1.0,
            project=PROJECT_DIR, name=name, exist_ok=True, verbose=False,
        )
        best = YOLO(f"{PROJECT_DIR}/{name}/weights/best.pt")
        v = best.val(data=f"{DATA_DIR}/data.yaml", split="test", imgsz=IMGSZ)

        # quick latency probe on the val images
        import numpy as np
        dummy = np.zeros((IMGSZ, IMGSZ, 3), dtype="uint8")
        for _ in range(10):
            best.predict(dummy, verbose=False)
        t = []
        for _ in range(50):
            t0 = time.perf_counter()
            best.predict(dummy, verbose=False)
            t.append((time.perf_counter() - t0) * 1000)
        rows.append({
            "model": m, "mAP50": round(v.box.map50, 4),
            "mAP50-95": round(v.box.map, 4),
            "mean_ms": round(np.mean(t), 1),
            "fps": round(1000 / np.mean(t), 1),
        })
    print(pd.DataFrame(rows).to_string(index=False))


## 9. Test on real cage frames (`cage_frames.zip` from Drive)
Extracts the zip, runs the model on a sample of real frames, and saves annotated images. Watch for the typical failure modes: huddled mice, mice under the hammocks, reflections, wall contact — these define what to oversample when labeling.

In [ ]:
import zipfile, glob, random

FRAMES_DIR = "/content/cage_frames"

if os.path.exists(ZIP_PATH):
    if not os.path.isdir(FRAMES_DIR) or not os.listdir(FRAMES_DIR):
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(FRAMES_DIR)
    # collect frames regardless of internal folder structure
    real_frames = sorted(
        f for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp")
        for f in glob.glob(f"{FRAMES_DIR}/**/{ext}", recursive=True)
    )
    print(f"Extracted {len(real_frames)} frames")

    # predict on a random sample of 30 frames, save annotated results
    model = YOLO(BEST)
    sample = random.sample(real_frames, min(30, len(real_frames)))
    model.predict(
        sample, conf=0.25, imgsz=IMGSZ, save=True,
        project=PROJECT_DIR, name="real_frames_test", exist_ok=True,
    )
    print(f"Annotated frames saved in {PROJECT_DIR}/real_frames_test/")
    print("Review them — note every failure mode you see.")
else:
    real_frames = []
    print(f"{ZIP_PATH} not found on Drive — check the path in cell 2.")


Extracted 131 frames

0: 480x640 (no detections), 3.4ms
1: 480x640 (no detections), 3.4ms
2: 480x640 (no detections), 3.4ms
3: 480x640 (no detections), 3.4ms
4: 480x640 (no detections), 3.4ms
5: 480x640 (no detections), 3.4ms
6: 480x640 (no detections), 3.4ms
7: 480x640 (no detections), 3.4ms
8: 480x640 1 mouse, 3.4ms
9: 480x640 (no detections), 3.4ms
10: 480x640 (no detections), 3.4ms
11: 480x640 (no detections), 3.4ms
12: 480x640 (no detections), 3.4ms
13: 480x640 (no detections), 3.4ms
14: 480x640 (no detections), 3.4ms
15: 480x640 (no detections), 3.4ms
16: 480x640 (no detections), 3.4ms
17: 480x640 (no detections), 3.4ms
18: 480x640 (no detections), 3.4ms
19: 480x640 (no detections), 3.4ms
20: 480x640 (no detections), 3.4ms
21: 480x640 1 mouse, 3.4ms
22: 480x640 (no detections), 3.4ms
23: 480x640 (no detections), 3.4ms
24: 480x640 (no detections), 3.4ms
25: 480x640 (no detections), 3.4ms
26: 480x640 (no detections), 3.4ms
27: 480x640 (no detections), 3.4ms
28: 480x640 (no detectio

### 10. FPS / latency benchmark
Warms up first, then reports mean and **p95** latency on real cage frames. Note: Colab T4 numbers ≠ your lab machine. Re-run this benchmarking code on the actual deployment hardware for the numbers that matter.

In [ ]:
import time
import numpy as np

model = YOLO(BEST)

# Use real cage frames if available, otherwise synthetic frames
if real_frames:
    bench_paths = random.sample(real_frames, min(300, len(real_frames)))
    frames = [cv2.imread(p) for p in bench_paths]
    frames = [f for f in frames if f is not None]
else:
    frames = [np.random.randint(0, 255, (480, 640, 3), dtype="uint8")
              for _ in range(300)]
print(f"Benchmarking on {len(frames)} frames")

for f in frames[:20]:                      # warmup
    model.predict(f, imgsz=IMGSZ, verbose=False)

times = []
for f in frames:
    t0 = time.perf_counter()
    model.predict(f, imgsz=IMGSZ, conf=0.25, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

times = np.array(times)
print(f"mean:  {times.mean():.2f} ms")
print(f"p50:   {np.percentile(times, 50):.2f} ms")
print(f"p95:   {np.percentile(times, 95):.2f} ms")
print(f"FPS (mean): {1000 / times.mean():.1f}")


Benchmarking on 131 frames
mean:  11.24 ms
p50:   10.93 ms
p95:   12.93 ms
FPS (mean): 89.0
